In [ ]:
import re
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score
)
from sklearn.metrics.pairwise import cosine_similarity

import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

 NLTK setup (needed for POS-tag based stylometric features)

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
from nltk import word_tokenize, sent_tokenize, pos_tag


In [ ]:
CONFIG = {
    "dataset_root": "reuters_extracted",
    "consolidated_csv": "reuters_50_50.csv",
    "chunk_size_words": 150,          # smaller chunks = more samples per author
    "chunk_overlap_words": 40,
    "embedding_model": "all-MiniLM-L6-v2",
    "min_chunk_words": 60,
    "top_k_stylo_features": 40,       # feature selection: keep only the best N stylometric features
    "sanity_check_n_authors": 5,
    "test_size_verification_pairs": 0.2      
}


Load data

In [ ]:
def load_raw_reuters(dataset_root: str) -> pd.DataFrame:
    """Fallback loader: reads directly from C50train / C50test folders."""
    records = []
    root = Path(dataset_root)
    for split_name in ["C50train", "C50test"]:
        split_dir = root / split_name
        if not split_dir.exists():
            continue
        for author_dir in sorted(split_dir.iterdir()):
            if not author_dir.is_dir():
                continue
            for txt_path in sorted(author_dir.glob("*.txt")):
                text = txt_path.read_text(encoding="utf-8", errors="ignore").replace("\r\n", "\n").strip()
                records.append({
                    "text": text,
                    "author": author_dir.name,
                    "split": "train" if split_name == "C50train" else "test",
                    "file_id": txt_path.stem,
                    "word_count": len(text.split()),
                })
    return pd.DataFrame(records)


csv_path = Path(CONFIG["consolidated_csv"])
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded consolidated CSV: {csv_path} ({len(df)} rows)")
else:
    df = load_raw_reuters(CONFIG["dataset_root"])
    print(f"Loaded from raw folders: {CONFIG['dataset_root']} ({len(df)} rows)")

df.head()

In [ ]:
print(f"Total articles: {len(df)}")
print(f"Unique authors: {df['author'].nunique()}")
print(f"Splits: {df['split'].value_counts().to_dict()}")
print(f"Avg words/article: {df['word_count'].mean():.1f}")
print(f"Min / Max words: {df['word_count'].min()} / {df['word_count'].max()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["word_count"].hist(bins=40, ax=axes[0], color="#6C63FF")
axes[0].set_title("Word Count Distribution")
axes[0].set_xlabel("Words per article")

df.groupby("author").size().sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="#6C63FF")
axes[1].set_title("Articles per Author")
axes[1].set_xticklabels([])
axes[1].set_xlabel("Author")

plt.tight_layout()
plt.show()

 Train-Test Author Distribution

In [ ]:
train_counts = df[df['split'] == 'train']['author'].value_counts()
test_counts = df[df['split'] == 'test']['author'].value_counts()

compare_df = pd.DataFrame({
    'Train': train_counts,
    'Test': test_counts
}).fillna(0).astype(int)

compare_df_sorted = compare_df.sort_values('Train', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
compare_df_sorted.plot(kind='bar', ax=ax, color=['#6C63FF', '#FF6B6B'])
ax.set_title('Author Distribution in Train vs Test Splits (Article Level)')
ax.set_xlabel('Author')
ax.set_ylabel('Number of Articles')
ax.legend(['Train', 'Test'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

Text Cleaning

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)      # collapse excess blank lines
    text = re.sub(r"[ \t]{2,}", " ", text)        # collapse excess spaces
    return text.strip()


df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(2)

Chunking

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if len(chunk_words) < 20:   # skip tiny trailing fragments
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_chunks_df(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        chunks = chunk_text(row["clean_text"], CONFIG["chunk_size_words"], CONFIG["chunk_overlap_words"])
        for i, chunk in enumerate(chunks):
            wc = len(chunk.split())
            if wc < CONFIG["min_chunk_words"]:
                continue
            records.append({
                "chunk_text": chunk,
                "author": row["author"],
                "split": row["split"],
                "source_file_id": row["file_id"],
                "chunk_id": f"{row['file_id']}_c{i}",
                "word_count": wc,
            })
    return pd.DataFrame(records)


chunks_df = build_chunks_df(df)
print(f"Total chunks: {len(chunks_df)}")
chunks_df.head()


Impact of Chunking (Distribution of chunk lengths)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
chunks_df['word_count'].hist(bins=30, color='#6C63FF', edgecolor='black')
plt.axvline(CONFIG['min_chunk_words'], color='red', linestyle='--', label=f"Min threshold ({CONFIG['min_chunk_words']})")
plt.xlabel('Words per Chunk')
plt.ylabel('Number of Chunks')
plt.title('Chunk Length Distribution')
plt.legend()

plt.subplot(1, 2, 2)
chunks_df.groupby('author')['word_count'].mean().sort_values().plot(kind='bar', color='#6C63FF')
plt.axhline(CONFIG['chunk_size_words'], color='red', linestyle='--', label=f'Chunk Size ({CONFIG["chunk_size_words"]})')
plt.xlabel('Author')
plt.ylabel('Average Chunk Length')
plt.title('Average Chunk Length per Author')
plt.xticks([])
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
# Define FUNCTION_WORDS (if not already defined)
FUNCTION_WORDS = [
    "the", "a", "an", "and", "but", "or", "so", "yet", "for", "nor",
    "in", "on", "at", "by", "with", "about", "against", "between", "into",
    "through", "during", "before", "after", "above", "below", "to", "from",
    "up", "down", "of", "off", "over", "under",
    "i", "you", "he", "she", "it", "we", "they", "this", "that", "these", "those",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "not", "no", "very", "just", "only", "also", "however", "therefore",
]

# Now your word‑cloud code:
function_words_set = set(FUNCTION_WORDS)
top_3_authors = df['author'].value_counts().head(3).index.tolist()
# ... rest of your wordcloud code

# ফাংশন ওয়ার্ডগুলোর উপর ভিত্তি করে ওয়ার্ডক্লাউড
function_words_set = set(FUNCTION_WORDS)

# টপ ৩ জন লেখক বেছে নিন (যাদের সবচেয়ে বেশি ডেটা আছে)
top_3_authors = df['author'].value_counts().head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, author in enumerate(top_3_authors):
    author_texts = ' '.join(chunks_df[chunks_df['author'] == author]['chunk_text'].tolist())
    words = word_tokenize(author_texts.lower())
    filtered_words = [w for w in words if w.isalpha() and w in function_words_set]
    text_for_cloud = ' '.join(filtered_words)
    
    wc = WordCloud(width=400, height=400, background_color='white', colormap='viridis').generate(text_for_cloud)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].axis('off')
    axes[idx].set_title(f'Author: {author} (Function Words)', fontsize=12)

plt.tight_layout()
plt.show()

Stylometric Feature Engineering

In [ ]:
import re
import textstat
import numpy as np
from collections import Counter
from nltk import word_tokenize, sent_tokenize, pos_tag

# --- Existing constants ---
FUNCTION_WORDS = [
    "the", "a", "an", "and", "but", "or", "so", "yet", "for", "nor",
    "in", "on", "at", "by", "with", "about", "against", "between", "into",
    "through", "during", "before", "after", "above", "below", "to", "from",
    "up", "down", "of", "off", "over", "under",
    "i", "you", "he", "she", "it", "we", "they", "this", "that", "these", "those",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "not", "no", "very", "just", "only", "also", "however", "therefore",
]

PUNCT_CHARS = [".", ",", ";", ":", "!", "?", "-", '"', "'", "(", ")"]

COMMON_TRIGRAMS = [
    "ing", "the", "ent", "ion", "and", "for", "her", "his", "tha", "was",
    "ere", "ate", "all", "you", "ear", "str", "ing", "men", "con", "pro"
]


def extract_stylometric_features(text: str) -> dict:
    words = word_tokenize(text)
    words_lower = [w.lower() for w in words if w.isalpha()]
    sentences = sent_tokenize(text)

    n_words = max(len(words_lower), 1)
    n_sentences = max(len(sentences), 1)
    sentence_lengths = [len(word_tokenize(sent)) for sent in sentences]

    features = {}

    # --- 1. Sentence & word structure ---
    features["avg_sentence_length"] = n_words / n_sentences
    features["avg_word_length"] = np.mean([len(w) for w in words_lower]) if words_lower else 0
    features["type_token_ratio"] = len(set(words_lower)) / n_words
    features["sentence_length_std"] = np.std(sentence_lengths) if sentence_lengths else 0

    # --- 2. Punctuation frequency (per 100 words) ---
    for p in PUNCT_CHARS:
        count = text.count(p)
        features[f"punct_{p}"] = (count / n_words) * 100

    features["punct_ellipsis"] = (text.count("...") / n_words) * 100
    features["punct_emdash"] = (text.count("—") / n_words) * 100

    # --- 3. Function word frequency (per 100 words) ---
    word_counts = Counter(words_lower)
    for fw in FUNCTION_WORDS:
        features[f"fw_{fw}"] = (word_counts.get(fw, 0) / n_words) * 100

    # --- 4. POS tag ratios ---
    try:
        tags = pos_tag(words)
        tag_counts = Counter(tag for _, tag in tags)
        total_tags = max(sum(tag_counts.values()), 1)
        pos_groups = {
            "pos_noun": ["NN", "NNS", "NNP", "NNPS"],
            "pos_verb": ["VB", "VBD", "VBG", "VBN", "VBP", "VBZ"],
            "pos_adj": ["JJ", "JJR", "JJS"],
            "pos_adv": ["RB", "RBR", "RBS"],
            "pos_pron": ["PRP", "PRP$"],
        }
        for group_name, tag_list in pos_groups.items():
            count = sum(tag_counts.get(t, 0) for t in tag_list)
            features[group_name] = count / total_tags
    except Exception:
        for group_name in ["pos_noun", "pos_verb", "pos_adj", "pos_adv", "pos_pron"]:
            features[group_name] = 0.0

    # --- 5. Misc habits ---
    features["avg_word_length_std"] = np.std([len(w) for w in words_lower]) if words_lower else 0
    features["capitalized_word_ratio"] = sum(1 for w in words if w.istitle()) / n_words
    features["uppercase_word_ratio"] = sum(1 for w in words if w.isupper() and len(w) > 1) / n_words

    # --- 6. Readability scores (using textstat) ---
    try:
        # Try to use textstat if available
        features["flesch_reading_ease"] = textstat.flesch_reading_ease(text)
        features["flesch_kincaid_grade"] = textstat.flesch_kincaid_grade(text)
        features["gunning_fog"] = textstat.gunning_fog(text)
        features["smog_index"] = textstat.smog_index(text)
        features["avg_syllables_per_word"] = textstat.avg_syllables_per_word(text)
    except Exception:
        # Fallback: approximate readability using avg word length and sentence length
        # This is a rough proxy (Flesch uses syllables – we estimate syllables ≈ 1 + floor(word_length/3))
        avg_word_len = features["avg_word_length"]
        avg_sent_len = features["avg_sentence_length"]
        # Estimated syllables per word (crude: 1.5 average for short words, but we use simple formula)
        # Actually we'll use a rough Flesch-like score: 206.835 - 1.015*(avg_sent_len) - 84.6*(avg_syllables/word)
        # Without true syllable count, we approximate syllables = 1 + avg_word_len/4 (this is a common approximation)
        est_syllables = 1 + avg_word_len / 4
        flesch = 206.835 - 1.015 * avg_sent_len - 84.6 * est_syllables
        flesch_kincaid = 0.39 * avg_sent_len + 11.8 * est_syllables - 15.59
        features["flesch_reading_ease"] = max(0, min(100, flesch))  # clamp to 0-100
        features["flesch_kincaid_grade"] = max(0, flesch_kincaid)
        features["gunning_fog"] = 0.4 * (avg_sent_len + 100 * (1 - features["type_token_ratio"]))
        features["smog_index"] = 1.043 * np.sqrt(max(0, 30 * (1 - features["type_token_ratio"])) + 3.1291)
        features["avg_syllables_per_word"] = est_syllables

    # --- 7. Character trigram frequencies (per 100 words) ---
    clean_text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    trigram_counts = {}
    for i in range(len(clean_text) - 2):
        tri = clean_text[i:i+3]
        if ' ' not in tri:
            trigram_counts[tri] = trigram_counts.get(tri, 0) + 1

    for tri in COMMON_TRIGRAMS:
        features[f"tri_{tri}"] = (trigram_counts.get(tri, 0) / n_words) * 100

    # --- 8. Stopword ratio (percentage of function words) ---
    stopwords = set(FUNCTION_WORDS)
    stopword_count = sum(1 for w in words_lower if w in stopwords)
    features["stopword_ratio"] = stopword_count / n_words

    return features

In [ ]:
feature_dicts = chunks_df["chunk_text"].apply(extract_stylometric_features)
features_df = pd.DataFrame(list(feature_dicts))
features_df.index = chunks_df.index

print(f"Stylometric feature matrix shape: {features_df.shape}")
features_df.head()


Stylometric Feature Correlation Matrix

In [ ]:
plt.figure(figsize=(18, 14))
corr = features_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, 
            linewidths=0.5, annot=False, square=True)
plt.title('Correlation Matrix of Stylometric Features')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Top 10 most correlated feature pairs:")
print(corr_pairs.head(10))

In [ ]:
per_author_means = features_df.groupby(chunks_df["author"]).mean()
feature_variance_across_authors = per_author_means.var().sort_values(ascending=False)

print("Top 10 most discriminative stylometric features (highest variance across authors):")
print(feature_variance_across_authors.head(10))

print("\nBottom 5 least discriminative (candidates to drop):")
print(feature_variance_across_authors.tail(5))

Author-wise Stylometric Profiles (Top 10 authors by article count)

In [ ]:
top_authors = df['author'].value_counts().head(10).index.tolist()
profile_df = features_df.loc[chunks_df['author'].isin(top_authors)]

selected_features = [
    'avg_sentence_length', 'type_token_ratio', 
    'pos_noun', 'pos_verb', 'pos_adj',
    'fw_the', 'fw_and', 'punct_.', 'punct_,', 'capitalized_word_ratio'
]

profile_means = profile_df[selected_features].groupby(chunks_df['author']).mean()
plt.figure(figsize=(14, 8))
sns.heatmap(profile_means, annot=True, cmap='viridis', fmt='.2f', cbar_kws={'label': 'Normalized Value'})
plt.title(f'Stylometric Profile Heatmap (Top {len(top_authors)} Authors)')
plt.xlabel('Features')
plt.ylabel('Author')
plt.tight_layout()
plt.show()

Semantic Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CONFIG["embedding_model"])
embeddings = embedder.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"Embedding matrix shape: {embeddings.shape}")

Combine Features

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
import numpy as np

scaler = StandardScaler()
stylo_scaled = scaler.fit_transform(features_df.values)

y_all = chunks_df["author"].values
label_encoder = LabelEncoder()
y_all_encoded = label_encoder.fit_transform(y_all)

selector = SelectKBest(score_func=f_classif, k=min(CONFIG["top_k_stylo_features"], stylo_scaled.shape[1]))
stylo_selected = selector.fit_transform(stylo_scaled, y_all_encoded)

selected_columns = features_df.columns[selector.get_support()]
print(f"Kept {len(selected_columns)} / {features_df.shape[1]} stylometric features:")
print(list(selected_columns))

X = np.hstack([stylo_selected, embeddings])
print(f"\nFinal feature matrix shape: {X.shape}")

In [ ]:
from wordcloud import WordCloud
# Define FUNCTION_WORDS (if not already defined)
FUNCTION_WORDS = [
    "the", "a", "an", "and", "but", "or", "so", "yet", "for", "nor",
    "in", "on", "at", "by", "with", "about", "against", "between", "into",
    "through", "during", "before", "after", "above", "below", "to", "from",
    "up", "down", "of", "off", "over", "under",
    "i", "you", "he", "she", "it", "we", "they", "this", "that", "these", "those",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "not", "no", "very", "just", "only", "also", "however", "therefore",
]

# Now your word‑cloud code:
function_words_set = set(FUNCTION_WORDS)
top_3_authors = df['author'].value_counts().head(3).index.tolist()
# ... rest of your wordcloud code

# ফাংশন ওয়ার্ডগুলোর উপর ভিত্তি করে ওয়ার্ডক্লাউড
function_words_set = set(FUNCTION_WORDS)

# টপ ৩ জন লেখক বেছে নিন (যাদের সবচেয়ে বেশি ডেটা আছে)
top_3_authors = df['author'].value_counts().head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, author in enumerate(top_3_authors):
    author_texts = ' '.join(chunks_df[chunks_df['author'] == author]['chunk_text'].tolist())
    words = word_tokenize(author_texts.lower())
    filtered_words = [w for w in words if w.isalpha() and w in function_words_set]
    text_for_cloud = ' '.join(filtered_words)
    
    wc = WordCloud(width=400, height=400, background_color='white', colormap='viridis').generate(text_for_cloud)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].axis('off')
    axes[idx].set_title(f'Author: {author} (Function Words)', fontsize=12)

plt.tight_layout()
plt.show()

Train/Test Split

In [ ]:
train_mask = chunks_df["split"].values == "train"
test_mask = chunks_df["split"].values == "test"

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y_all_encoded[train_mask], y_all_encoded[test_mask]

print(f"Train chunks: {X_train.shape[0]}")
print(f"Test chunks:  {X_test.shape[0]}")

## Baselines — Know What "Good" Actually Means

Before judging any model, establish the floor. Random guessing on 50 classes should land near 2%.
The majority-class baseline should be similarly low since classes are balanced.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
RANDOM_STATE = 42 
dummy_random = DummyClassifier(strategy="uniform", random_state=RANDOM_STATE)
dummy_random.fit(X_train, y_train)
random_acc = accuracy_score(y_test, dummy_random.predict(X_test))

dummy_majority = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy_majority.fit(X_train, y_train)
majority_acc = accuracy_score(y_test, dummy_majority.predict(X_test))

print(f"Random-guess baseline accuracy:   {random_acc:.4f}")
print(f"Majority-class baseline accuracy: {majority_acc:.4f}")
print(f"Number of classes: {len(label_encoder.classes_)}  (theoretical random chance = {1/len(label_encoder.classes_):.4f})")

## 5-Author Sanity Check

Before trusting the full 50-author result, confirm the pipeline works on an easy version of the
task. If this doesn't score well, the bug is in the pipeline, not the task difficulty.

In [ ]:
sanity_authors = list(label_encoder.classes_[:CONFIG["sanity_check_n_authors"]])
sanity_mask_train = np.isin(chunks_df["author"].values[train_mask], sanity_authors)
sanity_mask_test = np.isin(chunks_df["author"].values[test_mask], sanity_authors)

X_train_s, y_train_s = X_train[sanity_mask_train], y_train[sanity_mask_train]
X_test_s, y_test_s = X_test[sanity_mask_test], y_test[sanity_mask_test]

sanity_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
sanity_model.fit(X_train_s, y_train_s)
sanity_acc = accuracy_score(y_test_s, sanity_model.predict(X_test_s))

print(f"Sanity check — accuracy on {CONFIG['sanity_check_n_authors']} authors only: {sanity_acc:.4f}")
print("If this is high (>0.7), the pipeline works and the full 50-author task is just inherently harder.")

 Within-Author vs Cross-Author Cosine Similarity (on Test set embeddings)

In [ ]:
test_embeddings = embeddings[test_mask]
test_authors = chunks_df[test_mask]['author'].values
rng = np.random.default_rng(42)
indices = np.where(test_mask)[0]
sample_indices = rng.choice(indices, size=min(500, len(indices)), replace=False)
sample_embs = embeddings[sample_indices]
sample_auths = chunks_df['author'].values[sample_indices]

same_sims = []
diff_sims = []

for i in range(len(sample_embs)):
    for j in range(i+1, len(sample_embs)):
        sim = cosine_similarity(sample_embs[i].reshape(1, -1), sample_embs[j].reshape(1, -1))[0][0]
        if sample_auths[i] == sample_auths[j]:
            same_sims.append(sim)
        else:
            diff_sims.append(sim)
        if len(same_sims) > 100 and len(diff_sims) > 100:  
            break
    if len(same_sims) > 100 and len(diff_sims) > 100:
        break

plt.figure(figsize=(8, 6))
plt.boxplot([same_sims, diff_sims], label=['Same Author', 'Different Author'], patch_artist=True,
            boxprops=dict(facecolor='#6C63FF'), medianprops=dict(color='black'))
plt.title('Distribution of Cosine Similarities (Chunk Level)')
plt.ylabel('Cosine Similarity')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
from scipy import stats
if same_sims and diff_sims:
    t_stat, p_val = stats.mannwhitneyu(same_sims, diff_sims, alternative='greater')
    print(f"Mann-Whitney U Test: p-value = {p_val:.5f} (Same author similarity > Different author similarity?)")

##  Author Attribution — Model Training

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        {"C": [0.01, 0.1, 1, 10]},
    ),
    "Linear SVM": (
        LinearSVC(random_state=RANDOM_STATE, max_iter=5000),
        {"C": [0.01, 0.1, 1, 10]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [200, 400], "max_depth": [None, 30]},
    ),
}

tuned_results = {}
for name, (estimator, grid) in param_grids.items():
    print(f"Tuning {name}...")
    search = GridSearchCV(estimator, grid, cv=cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train, y_train)
    preds = search.best_estimator_.predict(X_test)
    acc = accuracy_score(y_test, preds)
    tuned_results[name] = {
        "model": search.best_estimator_,
        "best_params": search.best_params_,
        "cv_score": search.best_score_,
        "test_accuracy": acc,
        "preds": preds,
    }
    print(f"  Best params: {search.best_params_}")
    print(f"  CV accuracy: {search.best_score_:.4f}  |  Test accuracy: {acc:.4f}")


In [ ]:
best_model_name = max(tuned_results, key=lambda k: tuned_results[k]["test_accuracy"])
best_model = tuned_results[best_model_name]["model"]
best_preds = tuned_results[best_model_name]["preds"]
best_acc = tuned_results[best_model_name]["test_accuracy"]

print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(f"Random-guess baseline:     {random_acc:.4f}")
print(f"Majority-class baseline:  {majority_acc:.4f}")
print(f"Best tuned model:          {best_model_name}")
print(f"Best model test accuracy:  {best_acc:.4f}")
print(f"Improvement over random:   {best_acc / max(random_acc, 1e-9):.1f}x")
print("=" * 60)
print()
print(classification_report(y_test, best_preds, target_names=label_encoder.classes_, zero_division=0))

## Confusion Matrix

Which authors get confused with each other? This tells us whether the model is really learning
style, or leaning on topic overlap between certain writers.

In [ ]:
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Purples", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title(f"Confusion Matrix — {best_model_name} (tuned)")
plt.xlabel("Predicted Author")
plt.ylabel("True Author")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



## Document-Level Evaluation

A more realistic evaluation: aggregate chunk-level predictions (majority vote) back up to the
original article, since in the real world you usually classify a whole document, not one chunk.

In [ ]:
test_chunks = chunks_df[test_mask].copy()
test_chunks["pred_label"] = label_encoder.inverse_transform(best_preds)

doc_level = (
    test_chunks.groupby("source_file_id")
    .agg(true_author=("author", "first"),
         pred_author=("pred_label", lambda x: x.value_counts().idxmax()))
)
doc_accuracy = (doc_level["true_author"] == doc_level["pred_author"]).mean()
print(f"Document-level accuracy (majority vote across chunks): {doc_accuracy:.4f}")

In [ ]:
import json
import joblib
from sklearn.feature_selection import SelectKBest, f_classif


joblib.dump(scaler, "stylo_scaler_v2.joblib")
joblib.dump(selector, "stylo_feature_selector_v2.joblib")
joblib.dump(label_encoder, "author_label_encoder_v2.joblib")
joblib.dump(best_model, "author_attribution_model_v2.joblib")

pipeline_config = {
    "embedding_model": CONFIG["embedding_model"],
    "chunk_size_words": CONFIG["chunk_size_words"],
    "chunk_overlap_words": CONFIG["chunk_overlap_words"],
    "best_model_name": best_model_name,
    "best_model_params": tuned_results[best_model_name]["best_params"],
    "test_accuracy": float(best_acc),
    "document_level_accuracy": float(doc_accuracy),
    "random_baseline": float(random_acc),
    "selected_stylometric_features": list(selected_columns),
}
with open("pipeline_config_v2.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)

print("Saved v2 pipeline artifacts.")


## Author Verification

A different, arguably more useful task: given two texts, **did the same person write both?**
We build this using cosine similarity between chunk embeddings — same-author pairs should score
higher than different-author pairs.

In [ ]:
def build_verification_pairs(chunks_df: pd.DataFrame, embeddings: np.ndarray, test_mask: np.ndarray,
                              n_pairs: int, random_state: int = RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    test_idx = np.where(test_mask)[0]
    test_authors = chunks_df["author"].values[test_idx]

    pairs, labels = [], []
    n_each = n_pairs // 2

    # Same-author pairs
    by_author = {}
    for idx, author in zip(test_idx, test_authors):
        by_author.setdefault(author, []).append(idx)

    authors_with_multiple = [a for a, idxs in by_author.items() if len(idxs) >= 2]
    for _ in range(n_each):
        author = rng.choice(authors_with_multiple)
        i, j = rng.choice(by_author[author], size=2, replace=False)
        pairs.append((i, j))
        labels.append(1)

    # Different-author pairs
    for _ in range(n_each):
        a1, a2 = rng.choice(list(by_author.keys()), size=2, replace=False)
        i = rng.choice(by_author[a1])
        j = rng.choice(by_author[a2])
        pairs.append((i, j))
        labels.append(0)

    return pairs, np.array(labels)


# --- Fix: Convert fraction to integer number of pairs ---
test_size = np.sum(test_mask)                                      # total number of test chunks
n_pairs = int(CONFIG["test_size_verification_pairs"] * test_size)  # e.g. 0.2 * 6406 ≈ 1281
# Ensure at least 2 pairs (so that n_each >= 1)
if n_pairs < 2:
    n_pairs = 2

pairs, pair_labels = build_verification_pairs(
    chunks_df, embeddings, test_mask, n_pairs
)

pair_similarities = np.array([
    cosine_similarity(embeddings[i].reshape(1, -1), embeddings[j].reshape(1, -1))[0][0]
    for i, j in pairs
])

auc = roc_auc_score(pair_labels, pair_similarities)
print(f"Verification ROC-AUC: {auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(pair_labels, pair_similarities)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="#6C63FF", label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Author Verification — ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

# Pick a threshold that balances TPR/FPR (Youden's J statistic)
best_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[best_idx]
print(f"Suggested similarity threshold: {best_threshold:.4f}")


In [ ]:
import json

joblib.dump(scaler, "stylo_scaler.joblib")
joblib.dump(label_encoder, "author_label_encoder.joblib")
joblib.dump(selector, "stylo_feature_selector.joblib")
joblib.dump(best_model, "author_attribution_model.joblib")

pipeline_config = {
    "embedding_model": CONFIG["embedding_model"],
    "chunk_size_words": CONFIG["chunk_size_words"],
    "chunk_overlap_words": CONFIG["chunk_overlap_words"],
    "verification_threshold": float(best_threshold),
    "best_model_name": best_model_name,
    "stylometric_feature_columns": list(features_df.columns),
    "selected_stylometric_features": list(selected_columns), 
}
with open("pipeline_config.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)

print("Saved: stylo_scaler.joblib, author_label_encoder.joblib, author_attribution_model.joblib, pipeline_config.json")

In [ ]:
def featurize_text(text: str) -> np.ndarray:
    """Turn raw text into the same combined feature vector used in training.
    IMPORTANT: must apply the SAME selector used during training, or the
    feature count won't match what best_model expects.
    """
    stylo = extract_stylometric_features(text)
    stylo_vec = np.array([[stylo[col] for col in features_df.columns]])
    stylo_vec_scaled = scaler.transform(stylo_vec)
    stylo_selected = selector.transform(stylo_vec_scaled)   # <-- the missing step

    emb = embedder.encode([text], convert_to_numpy=True)
    combined = np.hstack([stylo_selected, emb])
    return combined


def verify_authorship(text_a: str, text_b: str, threshold: float = None):
    threshold = threshold or best_threshold
    emb_a = embedder.encode([text_a], convert_to_numpy=True)
    emb_b = embedder.encode([text_b], convert_to_numpy=True)
    similarity = cosine_similarity(emb_a, emb_b)[0][0]
    same_author = similarity >= threshold
    return {"similarity": float(similarity), "threshold": float(threshold), "same_author": bool(same_author)}

In [ ]:
def featurize_text(text: str) -> np.ndarray:
    stylo = extract_stylometric_features(text)
    stylo_vec = np.array([[stylo[col] for col in features_df.columns]])
    stylo_scaled = scaler.transform(stylo_vec)
    stylo_selected = selector.transform(stylo_scaled)
    emb = embedder.encode([text], convert_to_numpy=True)
    return np.hstack([stylo_selected, emb])


def predict_author(text: str, top_k: int = 3):
    vec = featurize_text(text)
    if hasattr(best_model, "predict_proba"):
        proba = best_model.predict_proba(vec)[0]
        top_idx = np.argsort(proba)[::-1][:top_k]
        return [(label_encoder.inverse_transform([i])[0], float(proba[i])) for i in top_idx]
    elif hasattr(best_model, "decision_function"):
        scores = best_model.decision_function(vec)[0]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(label_encoder.inverse_transform([i])[0], float(scores[i])) for i in top_idx]
    pred = best_model.predict(vec)[0]
    return [(label_encoder.inverse_transform([pred])[0], None)]


# Test
sample_text = test_chunks.iloc[0]["chunk_text"]
true_author = test_chunks.iloc[0]["author"]
print(f"True author: {true_author}")
for author, score in predict_author(sample_text):
    score_str = f"{score:.3f}" if score is not None else "n/a"
    print(f"  Predicted: {author} (score: {score_str})")

In [ ]:
vec = featurize_text(sample_text)
print(vec.shape)  # should output (424,)

In [ ]:
# --- Verification demo: compare two chunks from the same vs. different authors ---
same_author_rows = test_chunks[test_chunks["author"] == true_author]
if len(same_author_rows) >= 2:
    text_a = same_author_rows.iloc[0]["chunk_text"]
    text_b = same_author_rows.iloc[1]["chunk_text"]
    print("Same-author pair:", verify_authorship(text_a, text_b))

other_author_rows = test_chunks[test_chunks["author"] != true_author]
text_c = other_author_rows.iloc[0]["chunk_text"]
print("Different-author pair:", verify_authorship(sample_text, text_c))